### Probability Fundamentals for ML

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

1. Distribution Statistics Calculator

In [ ]:
def distribution_stats(data: np.ndarray) -> dict:
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)

    return {
        "mean": np.mean(data),
        "median": np.median(data),
        "variance": np.var(data),
        "std": np.std(data),
        "min": np.min(data),
        "max": np.max(data),
        "range": np.max(data) - np.min(data),
        "iqr": q3 - q1
    }


def print_stats(title, stats):
    print(f"\nDistribution Stats: {title}")
    for key, value in stats.items():
        print(f"{key.capitalize():<10}: {value:.4f}")


diabetes = load_diabetes()
X_diabetes = diabetes.data
feature_names = diabetes.feature_names

# BMI column
bmi_index = feature_names.index("bmi")
bmi_stats = distribution_stats(X_diabetes[:, bmi_index])
print_stats("BMI Feature", bmi_stats)

# Age column
age_index = feature_names.index("age")
age_stats = distribution_stats(X_diabetes[:, age_index])
print_stats("Age Feature", age_stats)

2. Normal Distribution Check

In [ ]:
print(f"{'Feature':<12} {'Within μ±1σ':<15} {'Within μ±2σ'}")

normal_scores = []

for i, feature in enumerate(feature_names):
    col = X_diabetes[:, i]

    mu = np.mean(col)
    sigma = np.std(col)

    within_1 = np.mean(np.abs(col - mu) <= sigma) * 100
    within_2 = np.mean(np.abs(col - mu) <= 2 * sigma) * 100

    normal_scores.append((feature, within_1, within_2))

    print(f"{feature:<12} {within_1:>8.2f}% {within_2:>14.2f}%")

print("\nExpected Normal Distribution:")
print("μ ± 1σ → 68.3%")
print("μ ± 2σ → 95.4%")


3. Naive Bayes Spam Filter

In [ ]:
P_SPAM = 0.25
P_HAM = 0.75

word_probs = {
    "offer": {"spam": 0.60, "ham": 0.05},
    "meeting": {"spam": 0.05, "ham": 0.35},
    "free": {"spam": 0.55, "ham": 0.02}
}


def naive_bayes_spam(words_present):
    spam_prob = P_SPAM
    ham_prob = P_HAM

    for word in word_probs:
        if word in words_present:
            spam_prob *= word_probs[word]["spam"]
            ham_prob *= word_probs[word]["ham"]
        else:
            spam_prob *= (1 - word_probs[word]["spam"])
            ham_prob *= (1 - word_probs[word]["ham"])

    total = spam_prob + ham_prob
    return spam_prob / total


print("\nNaive Bayes Spam Filter")

test_cases = [
    ["offer", "free"],
    ["meeting"],
    ["offer", "meeting", "free"]
]

for words in test_cases:
    prob = naive_bayes_spam(words)
    label = "SPAM" if prob > 0.5 else "HAM"

    print(f"Words {words} → P(spam) = {prob*100:.2f}% → {label}")

4. Classifier Calibration Analysis

In [ ]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

positive_probs = probs[y_test == 1]
negative_probs = probs[y_test == 0]

print(f"Avg P(positive) for true POSITIVE cases: {np.mean(positive_probs):.4f}")
print(f"Avg P(positive) for true NEGATIVE cases: {np.mean(negative_probs):.4f}")


# Histogram
plt.figure(figsize=(8, 5))

plt.hist(
    positive_probs,
    bins=20,
    alpha=0.6,
    label="True Positive"
)

plt.hist(
    negative_probs,
    bins=20,
    alpha=0.6,
    label="True Negative"
)

plt.xlabel("Predicted Probability")
plt.ylabel("Frequency")
plt.title("Classifier Calibration Analysis")
plt.legend()

plt.savefig("calibration_analysis.png")
plt.close()

print("[Saved calibration_analysis.png]")